In [ ]:
import pyodbc
import pandas as pd
from google.cloud import bigquery
import json
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Ruta al archivo de configuración
config_file_path = "C:\\Users\\TU_USUARIO\\ruta\\a\\config.json"  # Reemplaza con tu ruta
with open(config_file_path, 'r') as f:
    config = json.load(f)

# Establece el ID del proyecto de Google Cloud
project_id = 'TU_PROJECT_ID'    # Reemplaza con tu ID de proyecto
dataset_id = 'TU_DATASET_ID'    # Reemplaza con tu ID de conjunto de datos

# Establece la variable de entorno para las credenciales de Google Cloud
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = config['google']['credentials']

# Configura el proxy HTTP en la dirección IP y el puerto adecuados
os.environ['HTTPS_PROXY'] = config['network']['https_proxy']

# 3. Crear cliente de BigQuery
client = bigquery.Client(project=project_id)

# 4. Lista de tablas a exportar, mapeadas a sus nombres de hoja
# Reemplaza los nombres de tabla y hoja según tu proyecto
tablas = [
    ("NOMBRE_TABLA_1", "HOJA_1"),
    ("NOMBRE_TABLA_2", "HOJA_2"),
    ("NOMBRE_TABLA_3", "HOJA_3"),
]

# 5. Nombre de archivo dinámico basado en el mes anterior
hoy = datetime.today()
mes_anterior = hoy - relativedelta(months=1)

# Formato: Abr_25, May_25, etc.
meses_es = {
    1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Ago",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"
}
mes_str = meses_es[mes_anterior.month]
anio_str = str(mes_anterior.year)[-2:]  # últimos 2 dígitos del año

nombre_archivo = f"Reporte_{mes_str}_{anio_str}.xlsx"  # Reemplaza el prefijo según tu reporte
excel_output_path = f"C:\\Users\\TU_USUARIO\\ruta\\salida\\{nombre_archivo}"  # Reemplaza con tu ruta de salida

# 6. Leer cada tabla y escribir en su hoja correspondiente
with pd.ExcelWriter(excel_output_path, engine='xlsxwriter') as writer:
    for nombre_tabla, nombre_hoja in tablas:
        query = f"SELECT * FROM `{project_id}.{dataset_id}.{nombre_tabla}`"
        df = client.query(query).to_dataframe()
        df.to_excel(writer, sheet_name=nombre_hoja, index=False)

print("Exportación completada:", excel_output_path)